<a href="https://colab.research.google.com/github/a-i-git/Deep-Learning/blob/main/ANN_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv('/content/mnist_small.csv')
df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,1,0,0,0,0,0,0,0,0,0,...,69,12,0,0,0,0,0,0,0,0
5996,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5997,8,0,0,0,0,0,0,0,0,0,...,39,47,2,0,0,29,0,0,0,0
5998,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
from sklearn.model_selection import train_test_split
X=df.drop(columns=['label'],axis=1)
y=df.iloc[:,0].values
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [33]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train_scaled=sc.fit_transform(X_train)
X_test_scaled=sc.transform(X_test)

In [55]:
import torch
from torch.utils.data import Dataset,DataLoader
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels=torch.tensor(labels,dtype=torch.long)
  def __len__(self):
    return len(self.features)
  def __getitem__(self, index):
    return self.features[index],self.labels[index]

In [56]:
train_dataset=CustomDataset(X_train_scaled,y_train)

In [57]:
test_dataset=CustomDataset(X_test_scaled,y_test)

Mini-Batch GD

In [58]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

NN Model

In [59]:
import torch.nn as nn
class NNModel(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network=nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10),
        # nn.Softmax()
    )
  def forward(self,x):
    return self.network(x)


In [60]:
learning_rate=0.1
epochs=25

In [61]:
import torch.optim as optim
model=NNModel(X_train.shape[1])

criterion=nn.CrossEntropyLoss()

optimiser=optim.SGD(model.parameters(),lr=learning_rate)

In [62]:
for epoch in range(epochs):
  total_epoch_loss=0

  for batch_features,batch_labels in train_loader:
    outputs=model(batch_features) #Forward Pass

    loss=criterion(outputs,batch_labels) #Calculate Loss

    #Backpropagation
    optimiser.zero_grad()
    loss.backward()

    #Update weights
    optimiser.step()

    total_epoch_loss=total_epoch_loss+loss.item()

  avg_loader=total_epoch_loss/len(train_loader)
  print(f'Epoch {epoch} Loss:{avg_loader}')

Epoch 0 Loss:0.9035057149330775
Epoch 1 Loss:0.5288508609930674
Epoch 2 Loss:0.43769898504018784
Epoch 3 Loss:0.3771790358424187
Epoch 4 Loss:0.32937957003712653
Epoch 5 Loss:0.281445612659057
Epoch 6 Loss:0.25587666789690655
Epoch 7 Loss:0.22420582674443723
Epoch 8 Loss:0.20492684113482634
Epoch 9 Loss:0.18371953800320626
Epoch 10 Loss:0.15729294353475173
Epoch 11 Loss:0.14723403011759123
Epoch 12 Loss:0.12877598534648618
Epoch 13 Loss:0.11358588290090363
Epoch 14 Loss:0.11563311308311919
Epoch 15 Loss:0.12319716303298871
Epoch 16 Loss:0.09675458948438366
Epoch 17 Loss:0.0848364109142373
Epoch 18 Loss:0.06307501656624179
Epoch 19 Loss:0.08138241316191852
Epoch 20 Loss:0.04086812658701092
Epoch 21 Loss:0.03087808662947888
Epoch 22 Loss:0.028390244045294822
Epoch 23 Loss:0.03543754692499836
Epoch 24 Loss:0.04918721687359114


Evaluation Code

In [63]:
model.eval()

NNModel(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [65]:
total=0
correct=0

with torch.no_grad():# No gradients to be calculated while prediction
  for batch_features,batch_labels in test_loader:
    outputs=model(batch_features)

    # print(outputs)  # gives the probabilites of all the 10 classes

    _,predicted=torch.max(outputs,1) # Taking the maximum of all the probabilities

    total=total+batch_labels.shape[0]

    correct=correct+(predicted==batch_labels).sum().item()

print(correct/total)

0.8358333333333333
